Problem Statement

The objective of this project is to develop a deep learning system that can understand and classify textual data based on its sentiment. We will first use the IMDB movie review dataset as a controlled learning problem and progressively build and compare Simple RNN, LSTM, and GRU models using PyTorch.

The project will begin with text preprocessing, tokenization, sequence padding, and word embeddings. We will then train recurrent neural network models to classify movie reviews as positive or negative. The models will be evaluated and compared using appropriate performance metrics, training behavior, and error analysis.

After establishing a strong understanding of sequence modeling, the learned techniques will be applied to a more practical YouTube Comment Analyzer, where comments can be classified according to meaningful categories such as sentiment or toxicity, depending on the available dataset and labeling quality.

IMDB movie reviews = training/learning environment to understand RNNs, LSTMs, and GRUs.

Then we take what we learned and apply it to the real-world YouTube Comment Analyzer.

In [4]:
import random 
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
import torch
print("PyTorch version:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())#torch.cuda.is_available() → checks whether a GPU is available.

PyTorch version: 2.14.0+cpu
CUDA available: False


The GPU check matters because training neural networks can be much faster on a GPU.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
!pip install torch datasets

PyTorch is the deep-learning framework we use to build/train the RNN, while Hugging Face Datasets is a separate library we use to load the IMDB data.

you do not need to install huggingface_hub separately for what we are doing
You already ran:
pip install torch datasets

The datasets package uses huggingface_hub internally, so it is normally installed as a dependency.

In [5]:
import torch
from datasets import load_dataset

print("PyTorch version:", torch.__version__)

r:\C\PYTHON PROGRAMS\Deep Learning Project 4(RNN)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.14.0+cpu


In [6]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

5.0.1
1.30.0


In [7]:
#Load IMDB
dataset=load_dataset('stanfordnlp/imdb')
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Yes. stanfordnlp/imdb is the Hugging Face dataset identifier (repository address/name) for the IMDb dataset. It tells load_dataset() exactly which dataset repository to fetch.

In [9]:
print(dataset['train'][0])
print(dataset["train"][1])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

The IMDB dataset contains movie reviews with labels:
text → movie review
label → 0 (negative) or 1 (positive)

In [8]:
print(dataset['test'][24000])

{'text': "The movie is about a girl who's not going to a bonfire only because she's baby-sitting that night. Nothing weird about that, right? Until ... The phone rings. Until ... The phone rings again. And again ... And again. Those are not some stupid prank calls. This is for real. If you wanna see how the girl reacts, just watch the movie.<br /><br />Great atmosphere filled with scary sounds. Very well performed by young Camilla Belle who got the lead role. I see in her some great potential to become a good actress. This is more than only a decent thriller, I have no idea why it's so underrated. Anyway, on my opinion this movie deserves more than only 4/10. 24% of all voters rated the movie with 1. Get serious, people. You couldn't get a better thriller for a title like this.", 'label': 1}


Our project uses:
TRAIN
25,000 labeled reviews
        ↓
Learn

TEST
25,000 labeled reviews
        ↓
Evaluate

We first loaded the raw dataset because we needed to confirm what it contains. Now preprocessing is the next major stage.

Text preprocessing
What are we doing?

Our dataset currently contains:
"I rented I AM CURIOUS-YELLOW from my video store..."
PyTorch cannot send this English sentence directly into an RNN.

We first convert it into tokens:
"I rented this movie"
        ↓
["i", "rented", "this", "movie"]

Then later:
["i", "rented", "this", "movie"]
        ↓
[15, 428, 37, 892]

The numbers are the word IDs in our vocabulary.
Why are we doing this?

Because the RNN ultimately works with numerical tensors, not words.

So our preprocessing pipeline is:

Raw text
   ↓
Cleaning
   ↓
Tokenization
   ↓
Vocabulary
   ↓
Integer encoding
   ↓
Padding
   ↓
Tensor
   ↓
Embedding
   ↓
RNN

One important decision

For this project, I recommend that we build the tokenizer/vocabulary ourselves rather than using a ready-made NLP tokenizer.

In [27]:
text=dataset['train'][0]['text']
print(text)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [28]:
tex=dataset['train'][0]['label']
print(tex)

0


Now we will perform the first preprocessing operation: basic text cleaning.

Our reviews contain HTML tags such as:
<br /><br />
These are not meaningful words for sentiment classification, so we should remove them.

In [10]:
import re
text=dataset['train'][0]['text']
clean_text=re.sub(r"<br\s*/?>",' ',text)
print(clean_text)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.  The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.  What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot

Tokenization
We want to convert:
"I rented I AM CURIOUS-YELLOW from my video store"
into individual tokens:
["i", "rented", "i", "am", "curious-yellow", "from", "my", "video", "store"]

For our first implementation, we'll use a simple Python tokenizer so you understand what is happening rather than hiding it inside a library.

In [12]:
def tokenize(text):
    return text.lower().split()

tokens=tokenize(clean_text)
print(tokens[:20])

['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


Why .lower()?
These:
Movie
movie
MOVIE
should normally be treated as the same word.
So we convert everything to lowercase.

Why .split()?
.split() separates the text wherever there is whitespace:

"I love this movie"
        ↓
["i", "love", "this", "movie"]

This is our first simple tokenizer.

Later, when we build the vocabulary, we'll convert these tokens into integer IDs.

So the pipeline currently is:
Raw review
    ↓
Remove HTML
    ↓
Lowercase
    ↓
Split into tokens
    ↓
["i", "rented", "i", "am", ...]

We need:

25,000 raw reviews
        ↓
tokenize() on every review
        ↓
25,000 lists of tokens

In [13]:
# So first let's apply our tokenizer to the whole training set.
train_tokens=[tokenize(text) for text in dataset['train']['text']]
print("Number of reviews:",len(train_tokens))
print("First review tokens:",train_tokens[0][:20])

Number of reviews: 25000
First review tokens: ['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


train_tokens[0][:20] means:
train_tokens[0] → first review's complete token list
[:20] → take only the first 20 tokens

Now we will take all tokens from those 25,000 reviews, count how often each word appears, and then create our word-to-ID mapping.

We will do this in two stages:
1. Count word frequencies
2. Keep the most frequent words and assign IDs

For our project, we'll use a fixed vocabulary size such as 10,000 words so the model remains manageable.This means -- we won’t keep every unique word in the dataset.
Suppose there are 50,000 different words across the reviews. We choose only the 10,000 most frequently occurring words as our vocabulary.

Top 10,000 words → get their own IDs
Remaining rare words → <UNK>
This keeps the vocabulary smaller, so the model is faster and uses less memory.

You’re right that train_tokens contains 25,000 separate lists
every individual token inside every review is counted

In [29]:
#we’ll build the vocabulary from the tokenized training reviews only
#This is the step where every word gets its consistent integer ID.

#We are only counting right now. The actual word-to-number mapping comes immediately after this.
from collections import Counter
word_counts=Counter()
for tokens in train_tokens:
    word_counts.update(tokens) #update(tokens) = go through the tokens and add 1 to each word's frequency

print("Unique words:",len(word_counts))
print("Top 10 words:", word_counts.most_common(10))#gives the 10 most frequently occurring words/tokens, along with their counts.

#Counter() itself stores the words and their frequencies, so we don't need to create a separate dictionary.


#Your output
#Unique words: 251637
#means that across the 25,000 training reviews, there are 251,637 different tokens/words in our vocabulary before limiting it to the top 10,000.

#the, appeared 322,198 times across the entire 25,000-review training set.

Unique words: 251637
Top 10 words: [('the', 322198), ('a', 159953), ('and', 158572), ('of', 144462), ('to', 133967), ('is', 104171), ('in', 90527), ('i', 70480), ('this', 69714), ('that', 66292)]


Now comes the important part: Vocabulary
We cannot assign random IDs separately for every review. We need one common vocabulary for the entire training dataset.
For example:
<padded> → 0
<unk>    → 1
the      → 2
movie    → 3
good     → 4
bad      → 5
...
Then every review uses the same mapping.

Vocabulary = one common dictionary that assigns a unique number (ID) to each word.
So if: "the" → 2
then every occurrence of "the" in every review will be represented by 2.


What are <PAD> and <UNK>?
<PAD> = padding token. We use it to fill shorter reviews so all sequences have the same length.

<UNK> = unknown token. Used when a word is not present in our vocabulary (for example, a rare word that we decided not to include).

The important point is:<UNK> does not mean “gibberish.” It means “a valid word that our vocabulary does not contain.”

For example, suppose we keep only the 10,000 most frequent words:
the       → 2
movie     → 3
good      → 4

If "fantabulous" exists in English but is too rare and therefore was not included in our 10,000-word vocabulary, then:
fantabulous → <UNK> → 1
So <UNK> means unknown to our vocabulary, not unknown to the English language.

--Build the actual vocabulary--
Now we take those 251,637 unique words and keep only the 10,000 most frequent.

Then we assign IDs:
<PAD> → 0
<UNK> → 1
most frequent word → 2
next word → 3
...
10,000th word → ID

This is the step that turns our frequency counter into an actual word → number dictionary that we'll use to encode the reviews.

We will create a mapping like:

<PAD> → 0
<UNK> → 1
most frequent word → 2
2nd most frequent → 3
...

In [30]:
MAX_VOCAB_SIZE=10000

# Get the 10,000 most frequent words
most_common_words=word_counts.most_common(MAX_VOCAB_SIZE-2) #most_common_words contains words in frequency order.

# Create word → ID mapping

word_to_id={
    "<PAD>":0,
    "<UNK>":1
}

for idx,(word,count) in enumerate(most_common_words,start=2):
    word_to_id[word]=idx

print("Vocabulary Size:",len(word_to_id))
print("First 10 entries:",list(word_to_id.items())[:10])


Vocabulary Size: 10000
First 10 entries: [('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('a', 3), ('and', 4), ('of', 5), ('to', 6), ('is', 7), ('in', 8), ('i', 9)]


MAX_VOCAB_SIZE - 2 is because we already reserved 2 IDs:
0 → <PAD>
1 → <UNK>
So out of 10,000 total IDs, the actual words get 9,998 IDs.

start=2
means the first actual word gets ID 2 because:
<PAD> → 0
<UNK> → 1

word_to_id[word] = idx
inserts the word as the key and its ID as the value into the dictionary.

.items() → gets (word, ID) pairs

count is the word's frequency, but we're not using it here.you technically don't need count here because we aren't using the frequency anymore.

We need (word, count) only because most_common_words contains pairs like:
("the", 322198)
("a", 159953)

We can write _ instead of count to show we're intentionally ignoring it:
for idx, (word, _) in enumerate(most_common_words, start=2):

Here _ simply means: “There is a value here, but I don't need it.”

#numerical encoding. 
Right now each review is:['i', 'rented', 'i', 'am', 'curious-yellow', ...]

But the RNN needs numbers:
['i', 'rented', 'i', 'am']
        ↓
[9, 742, 9, 317]
We will create a function that looks up each word in word_to_id. If the word isn't there, it uses the <UNK> ID 1.

In [31]:
def encode_review(tokens):
    return [word_to_id.get(word,word_to_id['<UNK>']) for word in tokens]

encoded_review=encode_review(train_tokens[0])
print('First 20 tokens:')
print(train_tokens[0][:20])

print("\nEncoded:")
print(encoded_review[:20])

First 20 tokens:
['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']

Encoded:
[9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14]


.get() is a dictionary method used to safely get the value for a key.

word_to_id.get(word, word_to_id["<UNK>"])
means:Look for word in word_to_id. If it exists, return its ID; otherwise return the ID of <UNK> (which is 1).

SO, word_to_id["<UNK>"]
is not automatically used. It is only used when the word is missing.

In [32]:
#Encode the entire training set, Now we need to do the same for all 25,000 training reviews.

encoded_train=[encode_review(tokens) for tokens in train_tokens]

print("Number of encoded reviews:",len(encoded_train))
print("First encoded review:",encoded_train[0][:20])
print('First 3 review encode:',[review[:20] for review in encoded_train[0:3]])




Number of encoded reviews: 25000
First encoded review: [9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14]
First 3 review encode: [[9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14], [1179, 226, 1, 1, 7, 3, 1, 4, 2352, 9842, 1, 12, 141, 662, 48, 1942, 954, 3187, 22, 77], [46, 58, 6, 909, 242, 10, 585, 5, 24, 8, 2, 3263, 10, 24, 7, 246, 15, 32, 3837, 18]]


So encoded_train becomes:
[
  [9, 1486, 9, 226, 1, ...],   # review 1
  [2, 45, 78, 91, ...],        # review 2
  [15, 8, 34, ...],             # review 3
  ...
]

Notice that we still have different sequence lengths. That's the next problem we need to solve.
the next step will be:
❗ different lengths → same length
which is where padding comes in.

Our encoded reviews still have different lengths:
Review 1 → 218 IDs
Review 2 → 97 IDs
Review 3 → 341 IDs

But when we create batches for PyTorch, we need a common sequence length.
So we'll choose:
MAX_LEN = 200

Then:
< 200 → add 0 (<PAD>) at the beginning
> 200 → keep only the first 200 IDs
= 200 → unchanged

After padding:
Review 1 → 200 IDs
Review 2 → 200 IDs
Review 3 → 200 IDs
Then we can finally convert them into PyTorch tensors.

Why 200?
It's a design choice: large enough to retain substantial review context, but not so large that computation and memory become unnecessarily expensive.

In [33]:
MAX_LEN=200
def pad_sequence(sequence,max_len=MAX_LEN):
    if len(sequence) < max_len:
        return sequence + [word_to_id["<PAD>"]] * (max_len-len(sequence))
    else:
        return sequence[:max_len]

padded_train=[pad_sequence(sequence) for sequence in encoded_train]
print("Number of reviews:",len(padded_train))
print("Length of first review:", len(padded_train[0]))
print("First 20 IDs:", padded_train[0][:20])



Number of reviews: 25000
Length of first review: 200
First 20 IDs: [9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14]


word_to_id["<PAD>"] → 0
max_len - len(sequence) → how many 0s we need to add
* → repeats 0 that many times
+ → appends those 0s to the original sequence

Example:
sequence = [9, 15, 23]
max_len = 5
Then: 5 - 3 = 2

so: [9, 15, 23] + [0, 0]
  → [9, 15, 23, 0, 0]
So yes: we add 0s until the sequence reaches max_len.

Next step — Do the same preprocessing for the test set
This is important: we must use the exact same vocabulary we built from training data. We do not create a new vocabulary for the test data.

In [34]:
test_tokens = [tokenize(text) for text in dataset["test"]["text"]]

encoded_test = [encode_review(tokens) for tokens in test_tokens]

padded_test = [pad_sequence(sequence) for sequence in encoded_test]

print("Number of test reviews:", len(padded_test))
print("Length of first test review:", len(padded_test[0]))

Number of test reviews: 25000
Length of first test review: 200


eventually we'll have:
Training:
padded_train + train labels

Testing:
padded_test + test labels

Convert the data into PyTorch tensors, Right now:
padded_train → Python list of 25,000 sequences
padded_test  → Python list of 25,000 sequences

We now convert them into PyTorch tensors, because PyTorch models work with tensors.

In [35]:
#First get the labels:
train_labels=dataset['train']['label']
test_labels=dataset['test']['label']


In [36]:
X_train=torch.tensor(padded_train,dtype=torch.long)
y_train=torch.tensor(train_labels,dtype=torch.float32)

X_test=torch.tensor(padded_test,dtype=torch.long)
y_test=torch.tensor(test_labels,dtype=torch.float32)

X_train and X_test contain word IDs such as:
[9, 1486, 9, 226, 1, ...]

dtype=torch.long → tells PyTorch: these are integer IDs (like 9, 1486, 226), used to look up words in the Embedding layer.
dtype=torch.float32 → tells PyTorch: these are decimal/continuous numbers; here, the labels 0 and 1 are stored as floating-point values for the loss calculation.

In [22]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: torch.Size([25000, 200])
y_train shape: torch.Size([25000])
X_test shape: torch.Size([25000, 200])
y_test shape: torch.Size([25000])


This means: 25,000 reviews × 200 tokens

Now we move to the first real PyTorch data-pipeline step: Dataset and DataLoader.
Create PyTorch Dataset

We currently have:X_train → 25,000 × 200 integer IDs ,y_train → 25,000 labels
X_test  → 25,000 × 200 integer IDs
y_test  → 25,000 labels
But we don't want to give all 25,000 reviews to the model at once.

Instead, we want small batches, for example:
25,000 reviews
      ↓
batch of 32
batch of 32
batch of 32
...
That is what Dataset and DataLoader help us organize.

In [37]:
#First, create the Dataset
from torch.utils.data import TensorDataset

train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)

TensorDataset simply pairs the input and its corresponding label:
X_train[0] ↔ y_train[0]
X_train[1] ↔ y_train[1]
X_train[2] ↔ y_train[2]
...
So one dataset item is basically:
(review sequence, correct label)

In [24]:
print(train_dataset[0])

(tensor([   9, 1486,    9,  226,    1,   34,   57,  433, 1495,   77,    5,   35,
           2, 9840,   11, 3471,   12,   50,   12,   14,   82,  702,    8,    1,
           9,   81,  510,   11,   29,   82,   12,   14,    1,   31, 2443,    1,
          46,   12,  125,  737,    6, 2764,   10, 4218, 1935,   99,    3,  376,
           5,  129, 1127,    1,    9,   61,   62,    6,   67,   10,   16,    1,
          13,   93,  131,    7, 6447,  197,    3,  185, 4380,  659, 1669,  699,
        5601,   36,  438,    6,  781,  292,   53,   64,   43,  506,    8,  928,
          53,  438,    6, 1182,   42,    1,    6,  242,   45,  391,    5,  797,
          19,   48,    2,  952,    1,  199,   43,  721,  954, 1600,  130,   15,
           2, 3472,  408,    4, 1977, 1600,    8,    2, 2351, 7328,    8,  187,
        2170, 9841,    4, 2268,    1,    5,    1,   43,   59, 6083,   19, 9045,
          53,   41,  454,   17,   42,  659,    1,    1,    4, 1114,    1,   13,
        1063, 1068,   87,   43,    9,  

You should see two tensors:
(sequence of 200 IDs, label)

In [38]:
#Then create DataLoaders
from torch.utils.data import DataLoader
BATCH_SIZE=32

train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Why shuffle=True for training?
We want the training samples to be presented in a different order each epoch, which helps avoid the model relying on the original ordering of the training data.

Why shuffle=False for testing?
Testing is only evaluation, so there is no benefit to randomly rearranging the test samples.

In [39]:
#Create the DataLoader and inspect one batch
X_batch, y_batch = next(iter(train_loader))

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

X batch shape: torch.Size([32, 200])
y batch shape: torch.Size([32])


What does this mean?
32  → 32 reviews in one batch
200 → 200 token IDs per review

For example:
Review 1 → 200 IDs → label 0
Review 2 → 200 IDs → label 1
Review 3 → 200 IDs → label 0
...
This is exactly what will be passed into the model batch by batch, rather than all 25,000 reviews at once.

next(iter(train_loader)) means:
Take the first batch from train_loader.

Breakdown:
iter(train_loader)
→ creates an iterator that can go through batches one by one.

next(...)
→ asks that iterator for the next batch.

So:next(iter(train_loader))
→ gives you the first batch of 32 reviews and their 32 labels.

That’s why we used:
X_batch, y_batch = next(iter(train_loader))

It separates that first batch into:
X_batch → 32 reviews × 200 IDs
y_batch → 32 labels

In [21]:
#Create the Embedding layer
#Let's first define our model settings:
VOCAB_SIZE = 10000
EMBEDDING_DIM = 128
HIDDEN_SIZE = 64


In [25]:
embedding = torch.nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    padding_idx=0
)

num_embeddings=10000
→ Our vocabulary has 10,000 IDs, from 0 to 9999.

embedding_dim=128
→ Each word ID will be converted into a 128-dimensional vector.
One word becomes one vector of 128 numbers.

padding_idx=0
→ ID 0 is <PAD>, so PyTorch knows that this is our padding token.

Embedding Layer
Our batch currently looks like:

X_batch shape = [32, 200]
That means:
32 reviews
×
200 word IDs

But these IDs are just numbers such as:
[9, 1486, 9, 226, 1, ...]

We don't want the RNN to treat 1486 as “more meaningful” than 9 just because it is numerically larger.

So we use an Embedding layer.
Conceptually:
word ID
   ↓
Embedding lookup
   ↓
vector

For example:

the → 2 → [0.12, -0.41, 0.72, ...]
movie → 150 → [0.31, 0.08, -0.55, ...]

The embedding vectors are learned during training.

In [40]:
embedded=embedding(X_batch)

print("Input Shape:",X_batch.shape)
print("Embedded shape:",embedded.shape)

Input Shape: torch.Size([32, 200])
Embedded shape: torch.Size([32, 200, 128])


In [41]:
import torch.nn as nn

class SimpleRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__() #“First, initialize the basic PyTorch nn.Module part of this model.”

        self.embedding = nn.Embedding(  #we are actually making the embedding layer a part of the RNN model.
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,  #“Use 64 hidden units.”
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        last_hidden = hidden[-1]  #Take the final hidden state from hidden.

        x = self.fc(last_hidden) #This sends the 64-number H200 through the linear layer.

        return x  #“Give this output back to whoever called the model.”

    #last_hidden = hidden[-1]
    #means:
    #“Give me the final 64-number memory (H200) for each review in the batch.


    #For a batch of 32 reviews:
    #last_hidden → [32, 64]
                   # ↓
                  #Linear
                    #↓
                #x → [32, 1]


        #So each review gets one output number.
       #That number is called a logit. It is not yet the final 0/1 probability.

Remember our flow:
1 word
  ↓
Embedding
  ↓
128 numbers
  ↓
RNN
  ↓
64 hidden numbers

RNN receives 128 numbers for each word and produces a hidden state of 64 numbers.

nn.Module = PyTorch's base class for creating neural networks.
When you write:
class SimpleRNNModel(nn.Module):

you are saying:
“I am creating my own neural network, using PyTorch's neural-network structure.”
It gives your model useful PyTorch features like storing layers and learning their parameters.

So remember:
nn.Module = the basic template/framework for a PyTorch neural network.

So there is only one actual embedding layer in our trained model. The earlier embedding was just used to understand/check it.

self.rnn =It means:
“Store an RNN layer inside our model and call it rnn.”

And:
nn.RNN(...)
means:
“Create a standard RNN layer using PyTorch.”

Now this line: self.fc = nn.Linear(hidden_size, 1)

means:
Create a fully connected (Linear) layer that takes 64 numbers and produces 1 number.

Because:
hidden_size = 64

So:
H200
  ↓
64 numbers
  ↓
Linear layer
  ↓
1 number

Here:
hidden_size = 64 inputs
1 = 1 output

We use 1 output because this is a binary classification problem: positive or negative.
So the complete model is:

Word IDs
   ↓
Embedding
   ↓
128 numbers per word
   ↓
RNN
   ↓
64-number final hidden state (H200)
   ↓
Linear(64 → 1)
   ↓
1 number

Now we move to the forward() function — this is the part that tells Python how an input review passes through our model.

def forward(self, x):
Very simply:

forward() = “What should happen when I give data to this model?”
Here, x is our input:

x = batch of reviews

For example:
x shape = [32, 200]

meaning 32 reviews, each containing 200 word IDs.

Next line:
x = self.embedding(x)

This means:
Take those word IDs and pass them through the embedding layer.

So:

[32, 200]
    ↓
Embedding
    ↓
[32, 200, 128]

Next line:
output, hidden = self.rnn(x)

This simply means:
Send the embedded words through the RNN.

Before this line:
x = [32, 200, 128]

So we have 32 reviews × 200 words × 128 numbers per word.

After the RNN, PyTorch gives us two things:
output  → hidden state for every word
hidden  → final hidden state

For our model:
output → [32, 200, 64]
hidden → [1, 32, 64]

The important one for our classification is the final hidden state, which contains the H200 for each review.


output = all hidden states H1 → H200
hidden = final hidden state H200 ✅

In [46]:
VOCAB_SIZE = 10000
EMBEDDING_DIM = 128
HIDDEN_SIZE = 64

model = SimpleRNNModel(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_SIZE
)

print(model)

SimpleRNNModel(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (rnn): RNN(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [47]:
criterion=nn.BCEWithLogitsLoss()

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.001
)



Let's break this down:
torch.optim.Adam
→ We are choosing the Adam optimization algorithm.

model.parameters()
→ Tell Adam:

“These are the weights you are allowed to update.”

That includes weights from our:
Embedding
RNN
Linear layer

And:
lr=0.001
means learning rate.

It controls how large the weight updates are.

BCEWithLogitsLoss() does not predict 0 or 1.
Your model itself produces a logit, which can be any real number:

... -3.2, -1.5, 0.4, 2.1, 5.7 ...

Then:
logit
  ↓
sigmoid
  ↓
probability between 0 and 1
  ↓
threshold at 0.5
  ↓
class 0 or 1


What does BCEWithLogitsLoss actually do?
It takes:

logit + actual_label
and calculates how wrong the model is.

So:
criterion = nn.BCEWithLogitsLoss()
is for calculating the loss, not for converting the output into 0/1.


One very important point:
Logit is NOT 1 or 0.
Logit is the raw model output.
0 or 1 is the final class prediction

Loss tells us how bad the prediction is.
Optimizer changes the weights to make the prediction better.

In [33]:
#First Training Loop
#This is the part where your SimpleRNN actually starts learning.

#One epoch → yes, we go through all batches once.

#So:
#1 epoch = all training batches processed once

model.train()
total_loss=0


for X_batch,y_batch in train_loader:  #→ Our DataLoader gives us one batch at a time.
    optimizer.zero_grad()      #→ Clears gradients from the previous batch.

    logits=model(X_batch)   
    y_batch=y_batch.float().unsqueeze(1)
    loss=criterion(logits,y_batch) #→ Compare predictions with the actual labels and calculate the error.
    loss.backward()
    optimizer.step()  #→ Adam uses those gradients to update the model's weights.
    

    total_loss=total_loss + loss.item()
average_loss=total_loss / len(train_loader)
print("Training Loss:",average_loss)

    #loss.backward()→ PyTorch calculates the gradients showing how each trainable parameter contributed to the error.
     #loss.item() simply takes the numerical value out of the PyTorch loss tensor.

Training Loss: 0.6965344560420726


For example:
loss = tensor(0.63, grad_fn=...)
Then:
loss.item()
gives:
0.63

So we can do:
total_loss += loss.item()

We use .item() because total_loss is just a normal Python number—we don't need to keep the gradient information for it.

total_loss += loss.item()
adds the loss from every batch so that at the end we can calculate the average loss for the whole epoch.

len(train_loader)
gives the total number of batches in train_loader.
batch_size = 32
25,000 ÷ 32 ≈ 782 batches

model.train() → Tells PyTorch that the model is in training mode.

logits = model(X_batch) → Forward pass.
The reviews travel through:

IDs
 ↓
Embedding
 ↓
RNN
 ↓
Linear
 ↓
logits

Think of a gradient as a direction telling the model how to change its weights to reduce the error.
Very simply:

Model makes prediction
        ↓
Calculate loss (error)
        ↓
Gradient tells:
"Which way should I change the weights?"
        ↓
Optimizer changes the weights
Tiny example

Suppose a weight is:
weight = 0.50
The model makes a prediction and gets a high loss.

The gradient might say:
gradient = +0.3

That means, roughly:
“Reducing this weight would help reduce the loss.”
The optimizer then uses that information to adjust the weight.

You currently have:
logits  → [32, 1]
y_batch → [32]
But BCEWithLogitsLoss() requires them to have the same shape.
so that's why written  y_batch = y_batch.float().unsqueeze(1)

Your labels are: [32]
which means 32 values.

We change them to:
[32, 1]

so they match the model output:
logits → [32, 1]
labels → [32, 1]

Also, .float() is important because BCEWithLogitsLoss expects floating-point target values.

The basic loop is:
For each batch:
    ↓
Take reviews + labels
    ↓
Forward pass
    ↓
Calculate loss
    ↓
Clear old gradients
    ↓
Backpropagation
    ↓
Update weights

“Optimizer applied → weights are improved → hopefully the next batches/epochs have lower loss.”

Epoch 1: model sees all batches once and updates its weights.
Epoch 2: it sees all batches again, but now with the updated weights, so it should generally make better predictions and have lower loss.

Just remember: improvement is expected, but not guaranteed every single epoch.

Your first training epoch completed successfully.
Training Loss: 0.6884

Now we move to multiple epochs + accuracy tracking.

We already trained for 1 epoch. Next, we’ll let the model learn repeatedly and measure both loss and accuracy.

In [34]:
num_epochs=5
for epoch in range(num_epochs):
    model.train()

    total_loss=0
    correct=0
    total=0

    for X_batch,y_batch in train_loader:
        optimizer.zero_grad()
        logits=model(X_batch)
        y_batch=y_batch.float().unsqueeze(1)
        loss=criterion(logits,y_batch)

        loss.backward()
        optimizer.step()

        total_loss=total_loss + loss.item()

        predictions=(torch.sigmoid(logits)>=0.5).float() #This converts:logit → sigmoid probability → 0 or 1
        correct=correct + (predictions==y_batch).sum().item()
        total=total + y_batch.size(0)
    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Loss: {average_loss:.4f}, Accuracy: {accuracy:.4f}")
#.4f = show the number with 4 digits after the decimal.

#it's like (32,1) ,size(0) means:Give me the size of dimension 0. So it gives 32
#means:Add 32 to the total number of reviews processed.
#1 batch = 32 reviews
#0 + 32 = 32 reviews
#32 + 32 = 64 reviews
#64 + 32 = 96 reviews
#...

#At the end, total ≈ 25,000.
#total → individual reviews

Epoch 1/5
Loss: 0.6886, Accuracy: 0.5274
Epoch 2/5
Loss: 0.6741, Accuracy: 0.5523
Epoch 3/5
Loss: 0.6716, Accuracy: 0.5609
Epoch 4/5
Loss: 0.6412, Accuracy: 0.5987
Epoch 5/5
Loss: 0.6171, Accuracy: 0.6179


What does >= 0.5 mean? We first do:
torch.sigmoid(logits)

Suppose sigmoid gives:
0.8
0.3
0.6
0.1

We use: >= 0.5 to decide the class:
0.8 >= 0.5 → True  → 1
0.3 >= 0.5 → False → 0
0.6 >= 0.5 → True  → 1
0.1 >= 0.5 → False → 0

So 0.5 is our cutoff:
probability >= 0.5 → Positive (1)
probability <  0.5 → Negative (0)

After >= 0.5, Python/PyTorch gives Boolean values:
True
False
True
False

But our labels are:
1.0
0.0
1.0
0.0

So:.float()
converts:
True  → 1.0
False → 0.0

What is y_batch.size(0)?
Suppose:
y_batch.shape = [32, 1]

The dimensions are:
32 → number of reviews
1  → one label for each review

size(0) means:
Give me the size of dimension 0.

So:y_batch.size(0)
gives:32

Therefore: total += y_batch.size(0)
means:
Add 32 to the total number of reviews processed.

In [35]:
print(X_train.shape)
print(y_train.shape)

torch.Size([25000, 200])
torch.Size([25000])


So we have 25,000 reviews, each padded to length 200.

Create validation split
Since our current train_loader uses the whole training set, we should split the training data into train + validation.
Now we'll split these 25,000 into:

Training   → 20,000
Validation → 5,000

The validation data is not used to update the model's weights. It is only used to check how well the model performs on data it hasn't trained on.

One small but important point: since we already trained the model for 5 epochs using all 25,000 samples, we should restart/reinitialize the SimpleRNN before this proper train/validation experiment to avoid leakage from the validation split

Why? We trained the current model using all 25,000 reviews. Now that we're splitting those same 25,000 into:
20,000 → training
5,000  → validation
the model must start fresh so it has never learned from those 5,000 validation reviews.

Then also we'll create a new train_loader and val_loader.

In [55]:
from torch.utils.data import TensorDataset,DataLoader
from sklearn.model_selection import train_test_split

X_tr,X_val,y_tr,y_val=train_test_split(
    X_train,y_train,test_size=0.2,random_state=42
)

train_dataset=TensorDataset(X_tr,y_tr)
val_dataset=TensorDataset(X_val,y_val)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)

print(X_tr.shape)
print(X_val.shape)

torch.Size([20000, 200])
torch.Size([5000, 200])


#Create a fresh SimpleRNN
Use the same architecture you already built, but create a new instance so it starts with fresh weights.

For example, if your previous model was:
model = SimpleRNNModel(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=64
)
run the same model-creation code again, assigning it to model.

Then recreate the optimizer:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

We need a new optimizer too, because it keeps information about the model's parameters during training

In [37]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [38]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        logits = model(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Loss: {average_loss:.4f}, Accuracy: {accuracy:.4f}")

Epoch 1/5
Loss: 0.5834, Accuracy: 0.6399
Epoch 2/5
Loss: 0.5549, Accuracy: 0.6558
Epoch 3/5
Loss: 0.5310, Accuracy: 0.6724
Epoch 4/5
Loss: 0.5005, Accuracy: 0.6827
Epoch 5/5
Loss: 0.4901, Accuracy: 0.6897


The only important change from before is that train_loader now contains 20,000 reviews instead of 25,000.

So the fresh SimpleRNN has successfully learned from the 20,000 training reviews.
Now comes the important part: validation.

but we will NOT do:
loss.backward()
optimizer.step()
because validation data must not update the weights.
That will tell us how well the SimpleRNN performs on data it did not train on.

In [ ]:
#Now we’ll evaluate the 5,000 validation reviews.
#The key difference from training is: no weight updates.

model.eval()   #“We are not training now; we are only evaluating/testing the model.”
total_val_loss=0
correct=0
total=0

with torch.no_grad():#“Don't calculate/store gradients because we aren't learning right now.”
    for X_batch,y_batch in val_loader:  #for X_batch, y_batch in val_loader:
#we are now taking batches from the validation set, not from the previous training result.
        logits=model(X_batch)
        y_batch=y_batch.float().unsqueeze(1)
        loss=criterion(logits,y_batch)
        total_val_loss+=loss.item()
        predictions=(torch.sigmoid(logits)>=0.5).float()
        correct+=(predictions==y_batch).sum().item()
        total+=y_batch.size(0)

val_loss = total_val_loss / len(val_loader)
val_accuracy = correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")


Validation Loss: 0.6883
Validation Accuracy: 0.6116


In [40]:
#So now we only need to run the test evaluation loop using that existing test_loader
model.eval()

total_test_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        logits = model(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        total_test_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

test_loss = total_test_loss / len(test_loader)
test_accuracy = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.9676
Test Accuracy: 0.5160


Our proper approach from here is:

SimpleRNN
   ↓
LSTM
   ↓
GRU
   ↓
Compare them under the same setup
   ↓
Choose the strongest model
   ↓
Save THAT model
   ↓
Use the saved model in YouTube Comment Analyzer

Once the best model is saved, its weights stay fixed. Tomorrow, reopening VS Code does not change the saved model unless you retrain and overwrite it.

We already have the same input:Review → 200 word IDs

Our LSTM model will be:
Embedding → LSTM → Linear → Logit
Use the same dimensions we used for SimpleRNN:

In [43]:
#Build the LSTM model

class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embedding_dim=128,hidden_dim=64):
        super().__init__()

        self.embedding=nn.Embedding(vocab_size,embedding_dim)

        self.lstm=nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc=nn.Linear(hidden_dim,1)

    def forward(self,x):
        x=self.embedding(x)

        output,(hidden,cell)=self.lstm(x)
        last_hidden=hidden[-1]

        logits=self.fc(last_hidden)
        return logits

    #The LSTM internally handles the cell state and gates we just learned.

In [50]:
model2 = LSTMModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=64
)
print(model)

LSTMModel(
  (embedding): Embedding(10000, 128)
  (lstm): LSTM(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [51]:
optimizer=torch.optim.Adam(
    model2.parameters(),
    lr=0.001
)

In [52]:
num_epochs=5

for epoch in range(num_epochs):
    model2.train()

    total_loss=0
    correct=0
    total=0

    for X_batch,y_batch in train_loader:
        optimizer.zero_grad()

        logits=model2(X_batch)
        y_batch=y_batch.float().unsqueeze(1)

        loss=criterion(logits,y_batch)
        loss.backward()
        optimizer.step()

        total_loss=total_loss+ loss.item()
        predictions=(torch.sigmoid(logits)>=0.5).float()

        correct+=(predictions==y_batch).sum().item()
        total=total+y_batch.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Loss: {average_loss:.4f}, Accuracy: {accuracy:.4f}")

Epoch 1/5
Loss: 0.6933, Accuracy: 0.5130
Epoch 2/5
Loss: 0.6764, Accuracy: 0.5575
Epoch 3/5
Loss: 0.6082, Accuracy: 0.6834
Epoch 4/5
Loss: 0.5029, Accuracy: 0.7731
Epoch 5/5
Loss: 0.4263, Accuracy: 0.8183


logits = model2(X_batch)
you are calling the model object like a function. PyTorch automatically sends X_batch through the model's forward() method.

In [53]:
#Now do the LSTM validation evaluation, exactly like we did for SimpleRNN.

model2.eval()

total_val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in val_loader:

        logits = model2(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        total_val_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

val_loss = total_val_loss / len(val_loader)
val_accuracy = correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

Validation Loss: 0.5493
Validation Accuracy: 0.7530


model2.eval()       → evaluation mode
torch.no_grad()     → don't calculate gradients
no backward()       → don't learn
no optimizer.step() → don't change weights

In [54]:
model2.eval()
total_test_loss=0
correct=0
total=0

with torch.no_grad():
    for X_batch,y_batch in test_loader:
        logits=model2(X_batch)
        y_batch=y_batch.float().unsqueeze(1)

        loss=criterion(logits,y_batch)
        total_test_loss+=loss.item()
        predictions=(torch.sigmoid(logits)>=0.5).float()
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

test_loss = total_test_loss / len(test_loader)
test_accuracy = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")



Test Loss: 0.5555
Test Accuracy: 0.7512


This gives us the final LSTM test performance

In [17]:
#GRU

class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.gru(x)

        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits

In [51]:
model3 = GRUModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=64
)

print(model3)

GRUModel(
  (embedding): Embedding(10000, 128)
  (gru): GRU(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [52]:
optimizer = torch.optim.Adam(
    model3.parameters(),
    lr=0.001
)

In [53]:
num_epochs = 5

for epoch in range(num_epochs):

    model3.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        logits = model3(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Loss: {average_loss:.4f}, Accuracy: {accuracy:.4f}")

Epoch 1/5
Loss: 0.6905, Accuracy: 0.5316
Epoch 2/5
Loss: 0.5095, Accuracy: 0.7518
Epoch 3/5
Loss: 0.3072, Accuracy: 0.8740
Epoch 4/5
Loss: 0.2127, Accuracy: 0.9194
Epoch 5/5
Loss: 0.1411, Accuracy: 0.9511


In [56]:
model3.eval()

total_val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in val_loader:

        logits = model3(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        total_val_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

val_loss = total_val_loss / len(val_loader)
val_accuracy = correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

Validation Loss: 0.0886
Validation Accuracy: 0.9764


In [57]:
model3.eval()

total_test_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        logits = model3(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        total_test_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

test_loss = total_test_loss / len(test_loader)
test_accuracy = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.4236
Test Accuracy: 0.8385


We have completed the baseline comparison:

SimpleRNN → baseline
LSTM      → better
GRU       → best so far

So we will tune the GRU only.

The tuning process will be:
Current GRU
   ↓
Try a few sensible settings
   ↓
Compare validation performance
   ↓
Choose the best GRU
   ↓
Test it once
   ↓
Save the best model
   ↓
Use it for YouTube Comment Analyzer

One important thing: we'll select the best version using validation performance, not test performance. The test set stays for the final evaluation.

Now let's do the first tuning experiment: Dropout + Early Stopping on the GRU.

You do not tune the already-trained GRU by continuing from its old weights, because then you are no longer comparing a clean baseline; the previous training has already influenced the weights.

We create a fresh model for tuning so we can clearly distinguish the tuned experiment from the original GRU baseline.

In [19]:
#Now we’ll build the tuned GRU model with dropout.
class GRUModelTuned(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=64, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.gru(x)

        last_hidden = hidden[-1]

        last_hidden = self.dropout(last_hidden)

        logits = self.fc(last_hidden)

        return logits

In [22]:
model4 = GRUModelTuned(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=64,
    dropout=0.3
)

print(model4)

GRUModelTuned(
  (embedding): Embedding(10000, 128)
  (gru): GRU(128, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [43]:
optimizer = torch.optim.Adam(
    model4.parameters(),
    lr=0.001
)

What changed?
Previously:
GRU → Linear
Now:
GRU → Dropout → Linear

Dropout randomly temporarily disables some neurons during training, which helps reduce overfitting.
One important detail: during model4.eval(), dropout is automatically turned off, so the model uses all neurons for validation/test.

Now the next step is to train model4 with early stopping and track validation loss after every epoch. This will tell us whether the dropout-tuned GRU actually improves over our original GRU.

In [ ]:
num_epochs = 10

best_val_loss = float("inf")
patience = 2
counter = 0

for epoch in range(num_epochs):

    # -------- TRAINING --------
    model4.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        logits = model4(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    # -------- VALIDATION --------
    model4.eval()

    total_val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            logits = model4(X_batch)

            y_batch = y_batch.float().unsqueeze(1)

            loss = criterion(logits, y_batch)

            total_val_loss += loss.item()

            predictions = (torch.sigmoid(logits) >= 0.5).float()

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    val_loss = total_val_loss / len(val_loader)
    val_accuracy = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

    # -------- EARLY STOPPING --------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0
        #counter counts:
        #How many consecutive epochs have failed to improve validation loss?

        torch.save(model4.state_dict(), "best_gru.pt")

    else:

        counter += 1

        if counter >= patience:
            print("Early stopping triggered.")
            break

#torch.save() only saves the model at that moment. It does not mean “finish the program.”

For each epoch, this whole block runs:

1. Train for the entire epoch
2. Validate for the entire validation set
3. Print the results
4. Check early stopping
5. Either continue to next epoch OR stop

So yes, Epoch 1 fully runs first. Then the print happens. Then early stopping is checked. If it doesn't trigger, Epoch 2 starts.

range(10) means maximum 10 epochs, but the loop can stop earlier because of break.
And torch.save() only saves the best weights at that moment; it does not stop the loop.

In [24]:
model4.load_state_dict(torch.load("best_gru.pt"))
model4.eval()

GRUModelTuned(
  (embedding): Embedding(10000, 128)
  (gru): GRU(128, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [49]:
#Since model4 is already loaded with best_gru.pt, now run the final test evaluation:
#This will evaluate the saved best GRU checkpoint on the test set without retraining or changing its weights.

model4.eval()

total_test_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        logits = model4(X_batch)

        y_batch = y_batch.float().unsqueeze(1)

        loss = criterion(logits, y_batch)

        total_test_loss += loss.item()

        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

test_loss = total_test_loss / len(test_loader)
test_accuracy = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.4187
Test Accuracy: 0.8185


The flow is:
Yesterday:
Train GRU
   ↓
Best validation performance
   ↓
Save best weights → best_gru.pt

Today:
Load best_gru.pt
   ↓
GRU has the SAME saved weights
   ↓
Test data enters the network
   ↓
Network uses those SAME weights
   ↓
Final predictions → test accuracy

So the test set does not retrain the GRU. It only asks:

“With the final saved weights, how well does this model perform on completely separate test data?”

The original GRU was better on the test set:

Original GRU: 83.85%
Tuned GRU: 81.85%

So for the final application, the original GRU is currently the better choice. The dropout + early-stopping experiment was useful, but it did not improve the held-out test result.

In [ ]:
torch.save(model3.state_dict(), "best_gru_original.pt")

#Now we have preserved the exact original GRU model that achieved 83.85% test accuracy:
#That is the model we'll use for the YouTube Comment Analyzer.

In [59]:
model3.load_state_dict(torch.load("best_gru_original.pt"))
model3.eval()

GRUModel(
  (embedding): Embedding(10000, 128)
  (gru): GRU(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

Now we’ll build the single-comment prediction function around your saved original GRU.

In [60]:
def predict_sentiment(comment):


     # 1. Convert text into tokens
    tokens=tokenize(comment)

    # 2. Convert words into IDs
    encoded=[word_to_id.get(word,word_to_id['<UNK>']) for word in tokens]

    # 3. Pad/truncate to 200
    padded=pad_sequence(encoded)

    # 4. Convert to tensor and add batch dimension
    x=torch.tensor(padded,dtype=torch.long).unsqueeze(0)

    
    model3.eval()  #← put model in evaluation mode


    with torch.no_grad():
        logits=model3(x)   # 5. Send comment through the GRU

        # 6. Convert logit to probability
        probability=torch.sigmoid(logits).item()

    # 7. Convert probability to sentiment

    if probability >= 0.5:
        return "Positive", probability
    else:
        return "Negative", probability
    

#So yes, model3.eval() prepares the trained GRU for prediction, and then x is passed into the model.
#model3(x) → actually send x through the network

padded → [200]
torch.tensor(...) → PyTorch tensor
.unsqueeze(0) → [1, 200]

1 = one comment (one batch item)
200 = its 200 word IDs.

So yes, we add the batch dimension first, giving [1, 200].




So 0.5 or more → Positive, and below 0.5 → Negative.

In [61]:
comment = "This video is amazing!"
result = predict_sentiment(comment)

print(result)

('Negative', 0.4203154146671295)
